# **Importar Configurações**

In [0]:
%run ./00_setup_config

# Importar **Bibliotecas**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window

from delta.tables import DeltaTable

# **Definição de Tabelas**

In [0]:
CATALOGO = "workspace"

TABELA_BRONZE = f"{CATALOGO}.{SQUAD}.bronze_{TABELA}"

TABELA_SILVER = f"{CATALOGO}.{SQUAD}.silver_{TABELA}"

TABELA_QUARENTENA = f"{CATALOGO}.{SQUAD}.quarantine_{TABELA}"

In [0]:
#Verificação das Tabelas Definidas
#print(TABELA_BRONZE)
#print(TABELA_SILVER)

# **# Ler Dados da Camada Bronze**

In [0]:
bronze_df = spark.table(TABELA_BRONZE)

display(bronze_df.limit(10))

# **Implementação do Watermark**

In [0]:
if spark.catalog.tableExists(TABELA_SILVER):

    ultimo_processado = (
        spark.table(TABELA_SILVER)
        .agg(max("bronze_ingested_at"))
        .first()[0]
    )

    lote_df = bronze_df.filter(
        col("bronze_ingested_at") > ultimo_processado
    )

else:

    lote_df = bronze_df

In [0]:
#TESTE
#Verificação da Quantidade de Dados do Lote
#print(lote_df.count())

# **Criar Função de Conversão de Datas**


In [0]:
def nanos_para_timestamp(coluna):

    return (
        col(coluna) / 1000000000
    ).cast("timestamp")

In [0]:
#Teste nos dados submetidos a Função de Conversão de Data

#display(
 #   lote_df.select(
 #       nanos_para_timestamp("dt_pedido")
 #   )
#)

In [0]:
bronze_df.printSchema()

# **Aplicar Transformações Silver**

In [0]:
silver_df = (

    lote_df

    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )

    .withColumn(
        "id_pedido",
        trim(col("id_pedido"))
    )

    .withColumn(
        "id_cliente",
        trim(col("id_cliente"))
    )

    .withColumn(
        "status_pedido",
        trim(col("status_pedido"))
    )

    .withColumn(
        "dt_pedido",
        nanos_para_timestamp("dt_pedido")
    )

)

In [0]:
#Visualização dos Dados na Camada Silver
#display(silver_df)

# **Criar Regras Técnicas**

In [0]:
falha_pedido = col("id_pedido").isNull()

falha_cliente = col("id_cliente").isNull()

falha_valor = (
    col("valor_total").isNull()
    |
    (col("valor_total") <= 0)
)

falha_frete = (
    col("valor_frete") < 0
)

# **Gerar Coluna de Quarentena**

In [0]:
df_validado = (

    silver_df

    .withColumn(

        "quarantine_reason",

        concat_ws(

            "; ",

            when(
                falha_pedido,
                lit("ID_PEDIDO_NULO")
            ),

            when(
                falha_cliente,
                lit("CLIENTE_NULO")
            ),

            when(
                falha_valor,
                lit("VALOR_INVALIDO")
            ),

            when(
                falha_frete,
                lit("FRETE_INVALIDO")
            )
        )
    )

)

In [0]:
#Visualização Dos dados DF_VALIDADO

#display(
 #   df_validado.select(
#        "id_pedido",
#        "quarantine_reason"
#    )
#)

# **Separar Dados Válidos e Inválidos**

In [0]:
df_validos = (

    df_validado
    .filter(
        col("quarantine_reason") == ""
    )

)

In [0]:
df_quarentena = (

    df_validado
    .filter(
        col("quarantine_reason") != ""
    )

)

In [0]:
#VALIDAR QUANIDADE DE REGISTROS NAS TABELAS DF_VALIDOS E DF_QUARENTENA

#print(df_validos.count())

#print(df_quarentena.count())

# **Deduplicação**

In [0]:
janela = (

    Window

    .partitionBy(
        "id_pedido"
    )

    .orderBy(
        col("bronze_ingested_at").desc()
    )

)

In [0]:
df_validos = (

    df_validos

    .withColumn(
        "rn",
        row_number().over(janela)
    )

    .filter(
        col("rn") == 1
    )

    .drop("rn")

)

In [0]:
#VALIDAÇÃO DO DISPLAY 

#display(df_validos)

# **Criar Função MERGE**

In [0]:
def upsert_delta(
    df,
    tabela,
    chave
):

    if not spark.catalog.tableExists(tabela):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(tabela)
        )

        return

    delta_table = DeltaTable.forName(
        spark,
        tabela
    )

    (
        delta_table.alias("d")
        .merge(
            df.alias("o"),
            f"d.{chave} = o.{chave}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# **Gravar Silver**

In [0]:
upsert_delta(
    df_validos,
    TABELA_SILVER,
    "id_pedido"
)

In [0]:
##VALIDAÇÃO DOS DADOS VALIDADOS TABELA_SILVER

display(
   spark.table(TABELA_SILVER)
)

# **Gravar Quarentena**

In [0]:
upsert_delta(
    df_quarentena,
    TABELA_QUARENTENA,
    "id_pedido"
)

In [0]:
##VALIDAÇÃO DOS DADOS VALIDADOS TABELA_QUARENTENA

# display(
#   spark.table(
#       TABELA_QUARENTENA
#    )
#)

# **Auditoria Final**

In [0]:
print("===== RESUMO SILVER =====")

print(
    f"Registros recebidos: {lote_df.count()}"
)

print(
    f"Registros válidos: {df_validos.count()}"
)

print(
    f"Registros em quarentena: {df_quarentena.count()}"
)